In [43]:
from sqlalchemy import create_engine, MetaData, Table, Column, Integer, String, Float, ForeignKey
import os

from dotenv import load_dotenv
load_dotenv()


DATABASE_URL = (
    f"postgresql://{os.environ['POSTGRES_USER']}:"
    f"{os.environ['POSTGRES_PASSWORD']}@"
    f"{os.environ['POSTGRES_SERVER']}:"
    f"{os.environ['POSTGRES_PORT']}/"
    f"{os.environ['POSTGRES_DB']}"
)

In [15]:
engine = create_engine(DATABASE_URL, echo=True)

In [57]:
# Acts as a fascade to keep track of information necesary to create tables, keep track
# of tables, etc.
meta = MetaData()

# Create the table (not in db yet)
people = Table("Fake",
               meta,
               Column("id", Integer, primary_key=True),
               Column("name", String, nullable=False),
               Column("age", Integer)
              )

things = Table("FakeThings",
               meta,
               Column("id", Integer, primary_key=True),
               Column("description", String, nullable=False),
               Column("value", Float),
               Column("owner", Integer, ForeignKey("Fake.id"))
              ) 

# Create the table in the db
meta.create_all(engine)
conn.commit()

2026-05-08 16:58:34,204 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-08 16:58:34,206 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_class.relname = %(table_name)s AND pg_catalog.pg_class.relkind = ANY (ARRAY[%(param_1)s, %(param_2)s, %(param_3)s, %(param_4)s, %(param_5)s]) AND pg_catalog.pg_table_is_visible(pg_catalog.pg_class.oid) AND pg_catalog.pg_namespace.nspname != %(nspname_1)s
2026-05-08 16:58:34,208 INFO sqlalchemy.engine.Engine [cached since 1837s ago] {'table_name': 'Fake', 'param_1': 'r', 'param_2': 'p', 'param_3': 'f', 'param_4': 'v', 'param_5': 'm', 'nspname_1': 'pg_catalog'}
2026-05-08 16:58:34,212 INFO sqlalchemy.engine.Engine SELECT pg_catalog.pg_class.relname 
FROM pg_catalog.pg_class JOIN pg_catalog.pg_namespace ON pg_catalog.pg_namespace.oid = pg_catalog.pg_class.relnamespace 
WHERE pg_catalog.pg_cla

In [23]:
# Need to interact w/ the tables
conn = engine.connect()

insert_statement = people.insert().values(name="Sophie", age=20)

# Prepares the database for insert
conn.execute(insert_statement)

# Actually insert to db
conn.commit()

2026-05-08 16:37:35,963 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-08 16:37:35,966 INFO sqlalchemy.engine.Engine INSERT INTO "Fake" (name, age) VALUES (%(name)s, %(age)s) RETURNING "Fake".id
2026-05-08 16:37:35,968 INFO sqlalchemy.engine.Engine [cached since 188.4s ago] {'name': 'Sophie', 'age': 20}
2026-05-08 16:37:35,974 INFO sqlalchemy.engine.Engine COMMIT


In [41]:
select_statement = people.select().where(people.c.age > 0)
result = conn.execute(select_statement)
result.fetchall()

2026-05-08 16:44:05,079 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-08 16:44:05,081 INFO sqlalchemy.engine.Engine SELECT "Fake".id, "Fake".name, "Fake".age 
FROM "Fake" 
WHERE "Fake".age > %(age_1)s
2026-05-08 16:44:05,082 INFO sqlalchemy.engine.Engine [cached since 382.5s ago] {'age_1': 0}


[(2, 'Kyle', 60), (3, 'Sophie', 20), (1, 'Tod', 30)]

In [42]:
stmt = people.update().where(people.c.name == "Dan").values(name="Tod")
conn.execute(stmt)
conn.commit()

2026-05-08 16:46:35,129 INFO sqlalchemy.engine.Engine UPDATE "Fake" SET name=%(name)s WHERE "Fake".name = %(name_1)s
2026-05-08 16:46:35,130 INFO sqlalchemy.engine.Engine [cached since 166s ago] {'name': 'Tod', 'name_1': 'Dan'}
2026-05-08 16:46:35,133 INFO sqlalchemy.engine.Engine COMMIT


In [59]:
insert_people = people.insert().values([
    {"name": "Sean", "age": 30},
    {"name": "Clara", "age": 21},
    {"name": "Mario", "age": 45},
    {"name": "Claudia", "age": 95},
]
)

insert_things = things.insert().values(
    [
        {"owner": 1, "description": "Laptop", "value": 50.00},
        {"owner": 2, "description": "Speaker", "value": 50.00},
        {"owner": 3, "description": "Phone", "value": 50.00}
    ]
)

# conn.execute(insert_people)
# conn.commit()

conn.execute(insert_things)
conn.commit()

2026-05-08 16:59:37,980 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-05-08 16:59:37,983 INFO sqlalchemy.engine.Engine INSERT INTO "FakeThings" (description, value, owner) VALUES (%(description_m0)s, %(value_m0)s, %(owner_m0)s), (%(description_m1)s, %(value_m1)s, %(owner_m1)s), (%(description_m2)s, %(value_m2)s, %(owner_m2)s)
2026-05-08 16:59:37,985 INFO sqlalchemy.engine.Engine [no key 0.00520s] {'description_m0': 'Laptop', 'value_m0': 50.0, 'owner_m0': 1, 'description_m1': 'Speaker', 'value_m1': 50.0, 'owner_m1': 2, 'description_m2': 'Phone', 'value_m2': 50.0, 'owner_m2': 3}
2026-05-08 16:59:37,989 INFO sqlalchemy.engine.Engine COMMIT
